# Notebook 02 — NLP Pipeline

1. Address parsing: extract city/state/hub_type from raw center names
2. Route embeddings: sentence-transformers on corridor descriptions
3. Delay reason classification: BERT on synthetic delay-reason text
4. Validate coverage (≥90% city extraction success rate)

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.nlp.address_parser import parse_hub_name, parse_series, enrich_dataframe, KNOWN_CITIES
from src.nlp.route_embedder import build_corridor_embeddings, save_embeddings
from src.nlp.delay_classifier import generate_synthetic_dataset, fine_tune
print('NLP modules loaded')

In [ ]:
# ── 1. Address Parser ──
test_names = [
    'Bengaluru_Whitefield_Gateway_Hub',
    'Delhi_Okhla_Phase2_DC',
    'Mumbai (Bhiwandi) - Fulfillment Center',
    'Chennai_Ambattur_Sorting_SC',
    'Hyderabad_Shamshabad_LM_Hub',
    'Kolkata_Dankuni_Transit_Hub',
    'Pune_Chakan_GH',
    'Jaipur_Sitapura_WH_DC',
    'Lucknow_Amausi_Phase1_FC',
    'Nagpur_Butibori_SC',
]
results = [parse_hub_name(n) for n in test_names]
parsed_df = pd.DataFrame(results)
print(parsed_df[['raw','city','state','hub_type']])
success_rate = parsed_df['city'].notna().mean() * 100
print(f'\nCity extraction success rate: {success_rate:.1f}%')

In [ ]:
# Run on real dataset
import os
if os.path.exists('data/processed/delhivery_clean.parquet'):
    df = pd.read_parquet('data/processed/delhivery_clean.parquet')
    df_enriched = enrich_dataframe(df)
    print('Source city extraction:')
    print(df_enriched['src_city'].value_counts().head(15))
    print(f"\nOverall success rate: {df_enriched['src_city'].notna().mean()*100:.1f}%")
    df_enriched.to_parquet('data/processed/delhivery_enriched.parquet', index=False)
else:
    print('Run pipeline first: python -m src.data.pipeline')

In [ ]:
# ── 2. Route Embeddings ──
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

sample_routes = [
    'Delhi Gateway Hub to Mumbai Fulfillment Center via FTL in the morning',
    'Delhi Gateway Hub to Mumbai Fulfillment Center via Carting in the evening',
    'Bengaluru Whitefield Hub to Chennai Sorting Center via Carting in the evening',
    'Kolkata Dankuni Hub to Bhubaneswar Hub via Carting at night',
    'Hyderabad Shamshabad Hub to Chennai Hub via FTL in the morning',
    'Mumbai Bhiwandi to Pune Hub via Carting in the afternoon',
]

model = SentenceTransformer('all-MiniLM-L6-v2')
embs = model.encode(sample_routes)

# PCA visualisation
pca = PCA(n_components=2)
coords = pca.fit_transform(embs)

plt.figure(figsize=(10, 6), facecolor='#0e1117')
plt.scatter(coords[:,0], coords[:,1], s=100, c='#4f90ff')
for i, route in enumerate(sample_routes):
    plt.annotate(route[:40], (coords[i,0], coords[i,1]), fontsize=8, color='white')
plt.title('Route Embedding PCA (2D projection)', color='white')
plt.tight_layout()
plt.savefig('reports/02_route_embeddings_pca.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()

# Cosine similarities
from numpy.linalg import norm
for i in range(len(embs)):
    for j in range(i+1, len(embs)):
        cos = np.dot(embs[i], embs[j]) / (norm(embs[i]) * norm(embs[j]))
        print(f'sim({i},{j}) = {cos:.4f} | {sample_routes[i][:30]} ↔ {sample_routes[j][:30]}')

In [ ]:
# ── 3. Delay Reason Classifier ──
if os.path.exists('data/processed/delhivery_enriched.parquet'):
    df_enrich = pd.read_parquet('data/processed/delhivery_enriched.parquet')
    if 'delay_ratio' not in df_enrich.columns:
        df_enrich['delay_ratio'] = df_enrich['actual_time'] / df_enrich['osrm_time']
    synth = generate_synthetic_dataset(df_enrich.head(1000))
    print('Synthetic delay dataset label distribution:')
    print(synth['label_name'].value_counts())
    print('\nSample texts:')
    print(synth.head(3)[['text','label_name']].to_string())
    
    # Fine-tune BERT (set epochs=1 for quick demo)
    # fine_tune(synth, epochs=1)  # uncomment to actually train
    print('\n(Set fine_tune epochs > 1 for production training)')